<a href="https://colab.research.google.com/github/MorreyB/PC-Free/blob/main/Linux_Virtual_PC.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [19]:
!bash

bash: cannot set terminal process group (1025): Inappropriate ioctl for device
bash: no job control in this shell


/content# ^C


In [23]:
import os, subprocess, shutil, psutil, time, threading, torch, datetime, socket
from google.colab import drive, output
from IPython.display import HTML, display

# --- 1. INITIALIZE & MOUNT DRIVE ---
drive.mount('/content/drive', force_remount=True)
P_BASE = '/content/drive/MyDrive/colab_desktop_data'
os.makedirs(P_BASE, exist_ok=True)

# --- 2. AUTOMATION: BACKGROUND CLEANUP ---
def auto_cleanup_backups(keep=5):
    backup_root = os.path.join(P_BASE, 'session_backups')
    while True:
        if os.path.exists(backup_root):
            backups = sorted([d for d in os.listdir(backup_root) if os.path.isdir(os.path.join(backup_root, d))])
            if len(backups) > keep:
                for i in range(len(backups) - keep):
                    target = os.path.join(backup_root, backups[i])
                    print(f"፤ [Auto-Cleanup] Removing old backup: {backups[i]}")
                    shutil.rmtree(target)
        time.sleep(3600)

threading.Thread(target=auto_cleanup_backups, daemon=True).start()

# --- 3. SYSTEM DEPENDENCIES & GPU DRIVERS ---
print("፤ Installing Dependencies (Firefox, XFCE, VNC, VirtualGL, Flatpak)...")
subprocess.run("add-apt-repository -y ppa:mozillateam/ppa", shell=True, capture_output=True)
subprocess.run("apt-get update", shell=True, capture_output=True)
subprocess.run("apt-get purge -y firefox", shell=True, capture_output=True)
subprocess.run("apt-get install -y xfce4 xfce4-goodies tightvncserver novnc python3-websockify sudo firefox flatpak libnvidia-gl-535 libegl1 mesa-utils", shell=True, capture_output=True)
subprocess.run("flatpak remote-add --if-not-exists flathub https://flathub.org/repo/flathub.flatpakrepo", shell=True)

if not shutil.which("vglrun"):
    subprocess.run("wget https://sourceforge.net/projects/virtualgl/files/3.1/virtualgl_3.1_amd64.deb/download -O vgl.deb", shell=True, capture_output=True)
    subprocess.run("dpkg -i vgl.deb && apt-get install -f -y", shell=True, capture_output=True)

with open('/etc/ld.so.conf.d/00-nvidia-priority.conf', 'w') as f:
    f.write('/usr/lib/x86_64-linux-gnu/nvidia\n/usr/lib64-nvidia\n/usr/lib/x86_64-linux-gnu\n')
subprocess.run("ln -sf /usr/lib/x86_64-linux-gnu/libEGL_nvidia.so.0 /usr/lib/x86_64-linux-gnu/libEGL.so.1", shell=True)
subprocess.run("ldconfig", shell=True)
for i in range(10):
    if not os.path.exists(f'/dev/nvidia{i}'): subprocess.run(f'mknod -m 666 /dev/nvidia{i} c 195 {i}', shell=True)

# --- 4. SESSION MANAGEMENT ---
def save_session():
    ts = datetime.datetime.now().strftime('%Y%m%d_%H%M%S')
    bdir = os.path.join(P_BASE, 'session_backups', ts)
    os.makedirs(bdir, exist_ok=True)
    for src, name in [('/root/.vnc', 'vnc_config'), ('/root/.config/xfce4', 'xfce4_settings')]:
        if os.path.exists(src): shutil.copytree(src, os.path.join(bdir, name), dirs_exist_ok=True)
    print(f"፤ Session saved: {ts}")

def restore_session():
    backup_root = os.path.join(P_BASE, 'session_backups')
    if not os.path.exists(backup_root): return
    backups = sorted([d for d in os.listdir(backup_root) if d != 'latest'])
    if backups:
        target = os.path.join(backup_root, backups[-1])
        print(f"፤ Restoring from: {backups[-1]}")
        for dest, name in [('/root/.vnc', 'vnc_config'), ('/root/.config/xfce4', 'xfce4_settings')]:
            src = os.path.join(target, name)
            if os.path.exists(src):
                if os.path.exists(dest): shutil.rmtree(dest)
                shutil.copytree(src, dest)

# --- 5. START VNC ENGINE ---
subprocess.run("pkill -9 -f Xtightvnc|websockify|dbus", shell=True)
restore_session()
!rm -rf /tmp/.X* /tmp/.X11-unix /root/.vnc/*.pid /root/.vnc/*.log
os.environ['USER'] = 'root'
os.makedirs("/root/.vnc", exist_ok=True)
if not os.path.exists("/root/.vnc/passwd"):
    subprocess.run("echo password | vncpasswd -f > /root/.vnc/passwd && chmod 600 /root/.vnc/passwd", shell=True)
with open("/root/.vnc/xstartup", "w") as f:
    f.write("#!/bin/sh\nexport DISPLAY=:10\nexport VGL_DISPLAY=egl\ndbus-launch --exit-with-session startxfce4 &\n")
subprocess.run("chmod +x /root/.vnc/xstartup", shell=True)
subprocess.run("vncserver :10 -geometry 1920x1080 -depth 24", shell=True)
subprocess.Popen("websockify --web /usr/share/novnc/ 6080 127.0.0.1:5910", shell=True)

# --- 6. APP LAUNCHERS ---
def launch_firefox():
    env = os.environ.copy()
    env.update({"DISPLAY": ":10", "VGL_DISPLAY": "egl", "__GLX_VENDOR_LIBRARY_NAME": "nvidia", "LD_PRELOAD": "/usr/lib/x86_64-linux-gnu/libGL.so.1"})
    subprocess.Popen(["/opt/VirtualGL/bin/vglrun", "-d", "egl", "firefox", "--no-remote"], env=env)
    print("፤ Firefox starting...")

def launch_sober():
    env = os.environ.copy()
    env.update({"DISPLAY": ":10", "VGL_DISPLAY": "egl", "__GLX_VENDOR_LIBRARY_NAME": "nvidia"})
    subprocess.Popen(["/opt/VirtualGL/bin/vglrun", "-d", "egl", "flatpak", "run", "org.sober.Sober"], env=env)
    print("፤ Sober starting...")

final_link = "https://6080-gpu-t4-s-kkb-usw4a0-19ewuilmdlgl5-a.us-west4-0.prod.colab.dev/vnc.html?autoconnect=true&reconnect=true"
display(HTML(f'<div style="padding:15px; border:2px solid #4285f4; border-radius:10px;"><b>Desktop Link:</b> <a href="{final_link}" target="_blank">{final_link}</a></div>'))
print("፤ All components initialized.")

Mounted at /content/drive
፤ Installing Dependencies (Firefox, XFCE, VNC, VirtualGL, Flatpak)...
፤ Restoring from: 20260619_114346


፤ All components initialized.


In [21]:
launch_sober()

፤ Sober starting...


In [15]:
# This cell is now redundant and can be removed.

In [16]:
# This cell is now redundant and can be removed.

In [17]:
# To start apps, use the functions defined in the master cell above:
# launch_firefox()
# launch_sober()
# save_session()

In [6]:
save_session()

፤ Session saved: 20260619_114346


In [7]:
import os
backup_path = '/content/drive/MyDrive/colab_desktop_data/session_backups/'
if os.path.exists(backup_path):
    display(os.listdir(backup_path))
else:
    print('Backup directory not found.')

['20260619_110854', '20260619_114346']

In [8]:
import shutil
import os

backup_root = os.path.join(P_BASE, 'session_backups')
if os.path.exists(backup_root):
    # Get all subdirectories in the backup folder, excluding any non-timestamped files
    backups = sorted([d for d in os.listdir(backup_root) if os.path.isdir(os.path.join(backup_root, d))])

    if len(backups) > 1:
        oldest_backup = backups[0]
        target_path = os.path.join(backup_root, oldest_backup)
        print(f"፤ Deleting oldest backup: {oldest_backup}")
        shutil.rmtree(target_path)
        print("፤ Deletion complete.")
    else:
        print("፤ Only one or zero backups found. Skipping deletion to prevent data loss.")
else:
    print("፤ Backup directory not found.")

፤ Deleting oldest backup: 20260619_110854
፤ Deletion complete.
